In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [2]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

In [3]:
df = pd.read_csv(INTERIM_DIR / "atp_matches_1990_onward_base.csv")
df.shape

(112056, 17)

In [4]:
df["tourney_date"] = pd.to_datetime(df["tourney_date"], format = "%Y%m%d")
df = df.sort_values(["tourney_date", "tourney_id", "match_num"]).reset_index(drop = True)
df.head()

,tourney_id,tourney_name,surface,tourney_level,tourney_date,match_num,winner_id,winner_name,winner_age,loser_id,loser_name,loser_age,best_of,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,1990-339,Adelaide,Hard,A,1990-01-01,1,101843,Sergi Bruguera,18.9,101632,Per Henricsson,20.3,3,25.0,591.0,282.0,70.0
1,1990-339,Adelaide,Hard,A,1990-01-01,2,100714,Broderick Dyke,29.0,101096,Gary Muller,25.0,3,146.0,157.0,127.0,182.0
2,1990-339,Adelaide,Hard,A,1990-01-01,3,101529,Michael Stich,21.2,101232,Nicklas Kroon,23.9,3,113.0,215.0,71.0,290.0
3,1990-339,Adelaide,Hard,A,1990-01-01,4,100628,Michiel Schapers,30.2,101434,Petr Korda,21.9,3,68.0,297.0,80.0,271.0
4,1990-339,Adelaide,Hard,A,1990-01-01,5,101404,Thomas Muster,22.2,101387,Claudio Pistolesi,22.3,3,34.0,496.0,100.0,224.0


In [5]:
def expected_score(rating_a: float, rating_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((rating_b - rating_a)/ 400.0))

In [6]:
ratings = defaultdict(lambda: 1500.0)

winner_elo_before = []
loser_elo_before = []
winner_expected = []
loser_expected = []

K = 32.0

for row in df.itertuples(index = False) :
    w_id = row.winner_id
    l_id = row.loser_id

    w_rating = ratings[w_id]
    l_rating = ratings[l_id]

    w_exp = expected_score(w_rating, l_rating)
    l_exp = expected_score(l_rating, w_rating)

    winner_elo_before.append(w_rating)
    loser_elo_before.append(l_rating)
    winner_expected.append(w_exp)
    loser_expected.append(l_exp)

    ratings[w_id] = w_rating + K * (1.0 - w_exp)
    ratings[l_id] = l_rating + K * (0.0 - l_exp)

In [7]:
df["winner_elo"] = winner_elo_before
df["loser_elo"] = loser_elo_before
df["winner_exp"] = winner_expected
df["loser_exp"] = loser_expected

df[[
    "tourney_date",
    "winner_name",
    "loser_name",
    "winner_elo",
    "loser_elo",
    "winner_exp"
]].head(10)

,tourney_date,winner_name,loser_name,winner_elo,loser_elo,winner_exp
0,1990-01-01,Sergi Bruguera,Per Henricsson,1500.0,1500.0,0.5
1,1990-01-01,Broderick Dyke,Gary Muller,1500.0,1500.0,0.5
2,1990-01-01,Michael Stich,Nicklas Kroon,1500.0,1500.0,0.5
3,1990-01-01,Michiel Schapers,Petr Korda,1500.0,1500.0,0.5
4,1990-01-01,Thomas Muster,Claudio Pistolesi,1500.0,1500.0,0.5
5,1990-01-01,John Fitzgerald,Xavier Daufresne,1500.0,1500.0,0.5
6,1990-01-01,Mark Kratzmann,Jason Stoltenberg,1500.0,1500.0,0.5
7,1990-01-01,Paul Annacone,Bruno Oresar,1500.0,1500.0,0.5
8,1990-01-01,Mark Koevermans,Alexander Mronz,1500.0,1500.0,0.5
9,1990-01-01,Patrik Kuhnen,Paul Haarhuis,1500.0,1500.0,0.5


In [8]:
df.to_csv(PROCESSED_DIR / "atp_matches_1990_onward_with_elo.csv", index = False)